# Example 16.5 / 1: Solve SEND + MORE = MONEY and Test Uniqueness

Companion notebook for **16.5 Cryptarithms** in the AMPL lab Documentation, section "Solve SEND + MORE = MONEY and Test Uniqueness". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Solve SEND + MORE = MONEY and Test Uniqueness

The model has no data, so data.py only names the words. The run solves with HiGHS, reads `Digit` with `to_dict()`, and reconstructs the word values in Python as an independent check:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set LETTERS := {'S','E','N','D','M','O','R','Y'};  # distinct letters of SEND + MORE = MONEY
var Digit {LETTERS} integer >= 0, <= 9;  # digit assigned to each letter

subject to DifferentDigits:  # different letters get different digits
    alldiff {a in LETTERS} Digit[a];
subject to NoLeadingZero {a in {'S','M'}}:  # no word starts with zero
    Digit[a] >= 1;
subject to Addition:  # SEND + MORE = MONEY in place-value form
    1000*Digit['S'] + 100*Digit['E'] + 10*Digit['N'] + Digit['D']
  + 1000*Digit['M'] + 100*Digit['O'] + 10*Digit['R'] + Digit['E']
  = 10000*Digit['M'] + 1000*Digit['O'] + 100*Digit['N']
  + 10*Digit['E'] + Digit['Y'];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
words = ["SEND", "MORE", "MONEY"]

In [5]:
# run.py
ampl.solve(solver="highs", verbose=False)
print(ampl.solve_result)
digit = {a: round(v) for a, v in ampl.var["Digit"].to_dict().items()}
value = {w: int("".join(str(digit[a]) for a in w)) for w in words}
print(value, value["SEND"] + value["MORE"] == value["MONEY"])

solved
{'SEND': 9567, 'MORE': 1085, 'MONEY': 10652} True


### Another way in amplpy

A feasible solve proves that one mapping satisfies the arithmetic; it does not prove the puzzle has only one. The run below, with the same model.mod and data.py, adds the no-good cut `Exclude` with `ampl.eval` and counts mappings with the solve-and-exclude loop of N-Queens (AMPL lab documentation, section 16.1). It counts three times: with the model as written, after `drop NoLeadingZero`, and with the carry formulation in place of `Addition` (`drop Addition` after adding the column balances). `drop` and `restore` go through `ampl.eval` because amplpy has no method for them:

In [6]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [7]:
# data.py
words = ["SEND", "MORE", "MONEY"]

In [8]:
# run.py
ampl.eval("""
param nfound integer >= 0 default 0;  # solutions recorded so far
param Found {1..nfound, LETTERS};  # digits of each recorded solution
subject to Exclude {k in 1..nfound}:  # no-good cut: differ from every recorded mapping
    exists {a in LETTERS} Digit[a] != Found[k,a];
""")


def count_mappings():
    ampl.param["nfound"] = 0
    ampl.eval("reset data Found;")
    while True:
        ampl.solve(solver="highs", verbose=False)
        if ampl.solve_result != "solved":
            return int(ampl.param["nfound"].value()), ampl.solve_result
        digits = ampl.var["Digit"].to_dict()  # read before nfound changes the model
        k = int(ampl.param["nfound"].value()) + 1
        ampl.param["nfound"] = k
        for a, v in digits.items():
            ampl.param["Found"][k, a] = round(v)


print("positional model:", count_mappings())
ampl.eval("drop NoLeadingZero;")
print("without NoLeadingZero:", count_mappings())
ampl.eval("restore NoLeadingZero;")
ampl.eval("""
var Carry {1..4} binary;  # carry out of column k, units column first
subject to Column1: Digit['D'] + Digit['E'] = Digit['Y'] + 10*Carry[1];
subject to Column2: Digit['N'] + Digit['R'] + Carry[1] = Digit['E'] + 10*Carry[2];
subject to Column3: Digit['E'] + Digit['O'] + Carry[2] = Digit['N'] + 10*Carry[3];
subject to Column4: Digit['S'] + Digit['M'] + Carry[3] = Digit['O'] + 10*Carry[4];
subject to FinalCarry: Carry[4] = Digit['M'];
drop Addition;
""")
print("carry model:", count_mappings())

positional model: (1, 'infeasible')
without NoLeadingZero: (25, 'infeasible')
carry model: (1, 'infeasible')


### Pure AMPL: model.run

The uniqueness test in pure AMPL, with the same `model.mod`: record the first mapping, exclude it, and solve again:

In [9]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [10]:
# data.py
words = ["SEND", "MORE", "MONEY"]

In [11]:
%%ampl_eval
# model.run
param nfound integer >= 0 default 0;  # solutions recorded so far
param Found {1..nfound, LETTERS};  # digits of each recorded solution
subject to Exclude {k in 1..nfound}:  # no-good cut: differ from every recorded mapping
    exists {a in LETTERS} Digit[a] != Found[k,a];

option solver highs, solver_msg 0;
solve;
display solve_result, Digit;
let nfound := 1;
let {a in LETTERS} Found[1,a] := round(Digit[a]);
solve;
display solve_result;

solve_result = solved

Digit [*] :=
D  7
E  5
M  1
N  6
O  0
R  8
S  9
Y  2
;

solve_result = infeasible

